## Open government data provided by **opendata.swiss**

_Automatically generated Python starter code for dataset_ **1dddad61-699c-4414-bfcb-104f186737b4**


## Dataset

# **Observed and simulated snow profile data**


## Description

This data set includes information on all observed and simulated snow profiles that were used to train and validate the<br>random forest model described in Mayer et al. &#40;2022&#41;. The RF model was trained to assess snow instability from simulated snow stratigraphy. The data set contains observed snow profiles from the region of Davos &#40;DAV subset, 512 profiles&#41; and from all over Switzerland &#40;SWISS subset, 230 profiles&#41;. For each observed snow profile, there is a corresponding simulated profile which was obtained using meteorological input data for the numerical snow cover model SNOWPACK. The information on the observed snow profile contains a Rutschblock test result including the depth of the failure interface. As part of the study described in Mayer et al. &#40;2022&#41;, each observed snow profile was manually compared to its simulated counterpart and the simulated layer corresponding to the Rutschblock failure layer was identified. <br>The data are provided in the following form: one file each per observed and simulated snow profile &#40;2x512 files DAV, 2x230 files SWISS&#41;, two files &#40;1 file DAV, 1 file SWISS&#41; containing the observed information on snow instability, the allocation between observed and simulated failure layer, and all features extracted from the simulated weak layers that were used to develop the RF model.


## Dataset links

[View this dataset in the data catalogue](https://opendata.swiss/de/dataset/observed-and-simulated-snow-profile-data)


## Metadata

- **identifier:** 10.16904&#95;ENVIDAT.351@envidat
- **publisher.name:** EnviDat
- **organization.display&#95;name:** EnviDat: das Umwelt-Datenportal
- **organization.url:** https://www.envidat.ch/
- **maintainer:** 
- **maintainer&#95;email:** 
- **keywords:** rutschblock, snowpack-simulation, snow-profiles, snowpack, snow-stratigraphy, snow-instability, failure-layer
- **issued:** 2022-01-01 00:00:00
- **metadata&#95;created:** 2026-04-23T14:34:15.915355
- **metadata&#95;modified:** 2026-11-14T02:47:31.005057
- **Resource id:** 2db5d4ec-509c-49dd-8949-24151a848776
- **Resource name:** 
- **Resource package&#95;id:** 1dddad61-699c-4414-bfcb-104f186737b4
- **Resource description:** Resource Title: SWISS data set: Properties of observed snow profiles and simulated failure layers<br>    Size: 105662 bytes<br>    Format: text/csv<br>    URL: https://www.envidat.ch/dataset/f109f639-c6c8-46d6-82fd-6a5dbc2bb114/resource/8307de98-3d32-40de-97e8-e938a026cf2b/download/properties&#95;observed&#95;and&#95;simulated&#95;swiss.csv<br>    Description: For each profile pair from the SWISS data set, this file contains one row with information on the observed snow stability and all extracted features describing the simulated failure layers. Please note that only those profile pairs which fulfilled the similarity criteria &#40;column similarity&#95;obs&#95;sim = 1, 121 profiles&#41; were used to validate the RF model. The content of the columns is explained in Table 1 of the ReadMe file.
- **Resource issued:** 2022-01-01 00:00:00
- **Resource modified:** 
- **Resource license:** https://opendata.swiss/terms-of-use&#35;terms&#95;by&#95;ask
- **Resource rights:** https://opendata.swiss/terms-of-use&#35;terms&#95;by&#95;ask


## Imports and helper functions

This template targets pandas (3.x), seaborn (0.13+), matplotlib (3.x), and requests (2.x); packages are not installed or pinned by the notebook. Check the versions below when sharing results.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
print(f"pandas {pd.__version__}; seaborn {sns.__version__}")

In [ ]:
from io import BufferedReader
from typing import BinaryIO

import requests

USER_AGENT = "OpendataSwiss-StarterCode-Python/1.0"
HTTP_TIMEOUT = 30


def get_dataset(
    url: str | BinaryIO,
    *,
    sep: str | None = None,
    drop_empty_unnamed: bool = True,
    **kwargs: object,
) -> pd.DataFrame:
    """Read one CSV resource; preserve data and report parsing failures."""
    if isinstance(url, str) and url.startswith(("https://", "http://")):
        with requests.get(
            url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT, stream=True
        ) as response:
            response.raise_for_status()
            # Decode HTTP compression while streaming CSV bytes into pandas.
            response.raw.decode_content = True
            # Let the buffer finish reading after the HTTP connection reaches EOF.
            response.raw.auto_close = False
            with BufferedReader(response.raw) as source:
                return get_dataset(source, sep=sep, drop_empty_unnamed=drop_empty_unnamed, **kwargs)
    data = pd.read_csv(url, sep=sep, engine="python", dtype_backend="numpy_nullable", **kwargs)
    # Extra fields can make pandas silently consume leading values as an index.
    if kwargs.get("index_col") is None and not isinstance(data.index, pd.RangeIndex):
        raise pd.errors.ParserError(
            "CSV rows contain unexpected index fields. Check the delimiter and header; "
            "set index_col explicitly only if these fields are an intentional index."
        )
    if drop_empty_unnamed and len(data) > 0:
        # Blank CSV headers (often from trailing separators) become "Unnamed: n".
        unnamed = data.columns.astype(str).str.fullmatch(r"Unnamed: \d+(?:\.\d+)?")
        empty_unnamed = unnamed & data.isna().all()
        if empty_unnamed.any():
            print("Removed empty unnamed columns:", data.columns[empty_unnamed].tolist())
            data = data.loc[:, ~empty_unnamed]
    return data

## Load data

Selected resource: **2db5d4ec-509c-49dd-8949-24151a848776** (CSV). This notebook loads only this resource.

For CSV, the separator is guessed. If parsing looks wrong (especially for a single-column file), set `sep=","` or `sep=";"` in `get_dataset(...)`. Check inferred types, missing-value codes, decimal marks, and identifiers with leading zeros; use `dtype`, `na_values`, `decimal`, or `encoding` as needed.

Unexpected index inference raises a parsing error to prevent values shifting under the wrong headers. Check the delimiter and header, or set `index_col` explicitly for an intentional index.

Fully empty columns named `Unnamed: n` are removed on import. Named empty columns and unnamed columns containing values are kept. Use `drop_empty_unnamed=False` to retain every column, including intentionally named `Unnamed: n` fields.


In [ ]:
df = get_dataset("https://www.envidat.ch/dataset/f109f639-c6c8-46d6-82fd-6a5dbc2bb114/resource/8307de98-3d32-40de-97e8-e938a026cf2b/download/properties_observed_and_simulated_swiss.csv")

## Analyze data

Inspect before cleaning: these cells leave `df` unchanged. Duplicate rows and missing values may be meaningful; check the metadata before removing or filling them.


In [ ]:
print(f"{len(df):,} rows × {len(df.columns):,} columns")
print(f"{df.duplicated().sum():,} duplicate rows after the first occurrence")
df.info()
display(df.head())

In [ ]:
# Column types, completeness, and distinct non-missing values.
profile = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=True),
    }
).sort_values("missing_pct", ascending=False)
display(profile.round(1))

In [ ]:
# Show up to 20 columns with missing values; percentages use all rows.
missing = profile.loc[profile["missing"] > 0].head(20).rename_axis("column").reset_index()
if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, max(2, len(missing) * 0.3)))
    sns.barplot(data=missing, x="missing_pct", y="column", color="steelblue", errorbar=None, ax=ax)
    ax.set(xlim=(0, 100), xlabel="Missing values (%)", ylabel="", title="Missingness by column")
    fig.tight_layout()
    plt.show()
else:
    print("No missing values to plot (or the dataset has no rows).")

In [ ]:
# Summarize numeric and non-numeric columns, including pandas string types.
if not df.empty:
    display(df.describe(include="all").T)
else:
    print("No data to summarize.")

In [ ]:
# Plot up to four numeric columns. Exclude missing/infinite values only from plots.
numeric = df.select_dtypes(include="number")
for column in numeric.columns[:4]:
    values = numeric[column].dropna()
    values = values.loc[~values.isin([float("inf"), -float("inf")])]
    if not values.empty:
        fig, ax = plt.subplots(figsize=(7, 3))
        sns.histplot(x=values, bins=30, ax=ax)
        ax.set(title=column, xlabel=column, ylabel="Count")
        fig.tight_layout()
        plt.show()

In [ ]:
# Top 10 values for up to three categorical columns, including missing values.
categorical = df.select_dtypes(include=["string", "object", "category", "bool"])
for column in categorical.columns[:3]:
    print(column)
    display(df[column].value_counts(dropna=False).head(10).to_frame("count"))

### Next steps

Choose columns from `df.columns`: compare groups, inspect a time trend, or plot two numeric variables. Adapt the commented examples below; use the metadata to decide which comparisons make sense.


In [ ]:
# Group summary (replace column names):
# df.groupby("group", dropna=False)["value"].agg(["count", "mean", "median"])

# Parse a date explicitly, then inspect a time trend:
# dated = df.assign(date=pd.to_datetime(df["date"], format="%Y-%m-%d", errors="raise"))
# dated.groupby("date")["value"].mean().plot(ylabel="Mean value")

# Relationship between two numeric variables:
# sns.scatterplot(data=df, x="x_column", y="y_column", alpha=0.5)

**Questions about the data?** envidat@wsl.ch &#124; EnviDat
